# Non-entailment and Confabulation Analysis

This notebook analyzes steering results to classify reasoning patterns using OpenAI's O3 model.
We focus on successful steerings to measure how frequently they result in:
- **Non-entailment**: True premises, non-entailed conclusion
- **Confabulation**: False premises, entailed conclusion
- **Sound reasoning**: True premises, entailed conclusion  
- **Hallucination**: False premises, non-entailed conclusion

Based on the 2×2 framework from the blog post analysis.

## Setup and Imports

In [ ]:
import os
import sys
import json
import pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from typing import Dict, List, Tuple, Any, Optional
import time
from tqdm import tqdm

# Environment and API setup
from dotenv import load_dotenv
load_dotenv()

# OpenAI API
from openai import OpenAI

# Add parent directory for imports
sys.path.append(str(Path().parent))
from analysis.utils.data_loader import CacheDataLoader, DatasetProcessor, ResponseParser

# Set up plotting
plt.style.use('default')
sns.set_palette("husl")
%matplotlib inline

## Authentication and Configuration

In [ ]:
# Load OpenAI API key
openai_api_key = os.getenv('OPENAI_API_KEY')
if not openai_api_key:
    raise ValueError("OpenAI API key not found in .env file. Please add OPENAI_API_KEY=your_key_here")

# Initialize OpenAI client
client = OpenAI(api_key=openai_api_key)

print("✓ OpenAI client initialized")

## Data Loading Extensions

In [ ]:
class SteeringAnalyzer:
    """Extended analyzer for steering results classification."""
    
    def __init__(self, cache_dir: str = "../cache", client=None):
        self.loader = CacheDataLoader(cache_dir)
        self.client = client
        self.classification_cache = {}
        
    def load_successful_steerings(self, 
                                models: Optional[List[str]] = None,
                                datasets: Optional[List[str]] = None,
                                alpha_range: Optional[List[int]] = None) -> List[Dict]:
        """Load successful steering results across experiments."""
        
        experiments = self.loader.get_all_experiments()
        successful_steerings = []
        
        for exp in tqdm(experiments, desc="Loading experiments"):
            # Filter by model and dataset if specified
            if models and exp['model'] not in models:
                continue
            if datasets and exp['dataset'] not in datasets:
                continue
                
            # Load steering results
            steering_results = self.loader.load_steering_results(exp['path'])
            
            if not steering_results:
                continue
                
            # Process each alpha value
            for key, results in steering_results.items():
                if not results:
                    continue
                    
                # Parse alpha value from key (e.g., "alpha_2_no" -> 2)
                try:
                    alpha_part = key.split('_')[1]
                    alpha_val = int(alpha_part)
                    direction = key.split('_')[2]
                except (IndexError, ValueError):
                    continue
                    
                # Filter by alpha range if specified
                if alpha_range and abs(alpha_val) not in alpha_range:
                    continue
                
                # Extract successful steerings
                for result in results:
                    if isinstance(result, dict) and result.get('success', False):
                        steering_data = {
                            'model': exp['model'],
                            'dataset': exp['dataset'],
                            'experiment_hash': exp['experiment_hash'],
                            'alpha': alpha_val,
                            'direction': direction,
                            'question': self._extract_question(result.get('original_prompt', '')),
                            'steered_generation': self._extract_generation_text(result.get('steered_generation')),
                            'target_answer': result.get('target_answer', ''),
                            'original_answer': result.get('original_answer', ''),
                            'new_answer': result.get('new_answer', ''),
                            'category': result.get('category', '')
                        }
                        successful_steerings.append(steering_data)
        
        print(f"Found {len(successful_steerings)} successful steerings")
        return successful_steerings
    
    def _extract_question(self, prompt: str) -> str:
        """Extract the core question from the full prompt."""
        if not prompt:
            return ""
        
        # Look for common question patterns
        lines = prompt.split('\n')
        question_lines = []
        
        for line in lines:
            line = line.strip()
            if any(marker in line for marker in ['Q:', 'Question:', 'Is the following', 'Does the following']):
                question_lines.append(line)
            elif line.startswith('"') and line.endswith('"'):
                question_lines.append(line)
                
        return ' '.join(question_lines) if question_lines else prompt[:200] + "..."
    
    def _extract_generation_text(self, generation) -> str:
        """Extract text from generation (handle list format)."""
        if isinstance(generation, list) and len(generation) > 0:
            return generation[0]
        elif isinstance(generation, str):
            return generation
        else:
            return str(generation)

# Initialize analyzer
analyzer = SteeringAnalyzer(client=client)
print("✓ Steering analyzer initialized")

## Load Available Experiments

In [ ]:
# Get overview of available experiments
experiments = analyzer.loader.get_all_experiments()

print(f"Total experiments found: {len(experiments)}")
print("\nModels available:")
models = sorted(set(exp['model'] for exp in experiments))
for model in models:
    print(f"  - {model}")

print("\nDatasets available:")
datasets = sorted(set(exp['dataset'] for exp in experiments))
for dataset in datasets:
    print(f"  - {dataset}")

# Check which experiments have steering results
experiments_with_steering = []
for exp in experiments:
    steering_results = analyzer.loader.load_steering_results(exp['path'])
    if steering_results:
        experiments_with_steering.append(exp)

print(f"\nExperiments with steering results: {len(experiments_with_steering)}")

## OpenAI O3 Classification System

In [ ]:
class ReasoningClassifier:
    """Classifier for reasoning patterns using OpenAI O3."""
    
    def __init__(self, client):
        self.client = client
        self.system_prompt = self._create_system_prompt()
        
    def _create_system_prompt(self) -> str:
        return """You are an expert at analyzing reasoning patterns in language model outputs.

Your task is to classify reasoning based on a 2×2 framework with these categories:

1. **Sound Reasoning**: The model states TRUE premises and reaches a conclusion that IS logically entailed by those premises.

2. **Non-entailment**: The model states TRUE premises but reaches a conclusion that is NOT logically entailed by those premises.

3. **Confabulation**: The model states FALSE premises but reaches a conclusion that IS logically entailed by those false premises.

4. **Hallucination**: The model states FALSE premises and reaches a conclusion that is NOT logically entailed by those premises.

Instructions:
- Focus on the factual accuracy of premises and logical validity of reasoning steps
- If you cannot confidently classify the reasoning, respond with "N/A"
- Provide a confidence level (high/medium/low) and brief reasoning for your classification
- Return your response as a JSON object with keys: "classification", "confidence", "reasoning"

Valid classifications: "sound_reasoning", "non_entailment", "confabulation", "hallucination", "N/A"
Valid confidence levels: "high", "medium", "low"""
    
    def classify_reasoning(self, question: str, steered_generation: str, max_retries: int = 3) -> Dict[str, str]:
        """Classify a single reasoning example."""
        
        user_prompt = f"""Question: {question}

Model's reasoning and answer: {steered_generation}

Please classify this reasoning pattern according to the framework provided."""
        
        for attempt in range(max_retries):
            try:
                response = self.client.chat.completions.create(
                    model="o1-preview",  # Use O1 as O3 may not be available yet
                    messages=[
                        {"role": "system", "content": self.system_prompt},
                        {"role": "user", "content": user_prompt}
                    ],
                    temperature=0.1,
                    max_tokens=500
                )
                
                result_text = response.choices[0].message.content
                
                # Try to parse JSON response
                try:
                    result = json.loads(result_text)
                    
                    # Validate required fields
                    required_fields = ['classification', 'confidence', 'reasoning']
                    if all(field in result for field in required_fields):
                        return result
                    else:
                        print(f"Missing fields in response: {result}")
                        
                except json.JSONDecodeError:
                    # Try to extract classification from text if JSON parsing fails
                    classification = self._extract_classification_from_text(result_text)
                    if classification:
                        return {
                            'classification': classification,
                            'confidence': 'medium',
                            'reasoning': result_text[:200] + "..."
                        }
                
                # If we get here, something went wrong
                print(f"Attempt {attempt + 1} failed, response: {result_text[:100]}...")
                time.sleep(1)  # Brief pause before retry
                
            except Exception as e:
                print(f"API error on attempt {attempt + 1}: {e}")
                if attempt < max_retries - 1:
                    time.sleep(2 ** attempt)  # Exponential backoff
                    
        # If all retries failed
        return {
            'classification': 'N/A',
            'confidence': 'low',
            'reasoning': 'Classification failed after multiple attempts'
        }
    
    def _extract_classification_from_text(self, text: str) -> Optional[str]:
        """Fallback method to extract classification from non-JSON response."""
        text_lower = text.lower()
        
        classifications = [
            'sound_reasoning', 'non_entailment', 'confabulation', 'hallucination', 'n/a'
        ]
        
        for classification in classifications:
            if classification.replace('_', ' ') in text_lower or classification in text_lower:
                return classification
        
        return None

# Initialize classifier
classifier = ReasoningClassifier(client)
print("✓ Reasoning classifier initialized")

## Test Classification on Sample Data

In [ ]:
# Load a small sample of successful steerings for testing
sample_steerings = analyzer.load_successful_steerings(
    models=['Qwen_Qwen2.5-3B-Instruct'],  # Focus on one model for testing
    datasets=['sports_understanding'],     # Focus on one dataset
    alpha_range=[2, 4, 6]                 # Test with moderate alpha values
)

print(f"Sample size for testing: {len(sample_steerings)}")

if sample_steerings:
    print("\nFirst sample:")
    sample = sample_steerings[0]
    for key, value in sample.items():
        if key == 'steered_generation':
            print(f"{key}: {value[:200]}...")
        else:
            print(f"{key}: {value}")

In [ ]:
# Test classification on a single example
if sample_steerings:
    test_sample = sample_steerings[0]
    
    print("Testing classification...")
    print(f"Question: {test_sample['question']}")
    print(f"Generation: {test_sample['steered_generation'][:300]}...\n")
    
    result = classifier.classify_reasoning(
        test_sample['question'],
        test_sample['steered_generation']
    )
    
    print("Classification result:")
    print(json.dumps(result, indent=2))

## Batch Processing with Caching

In [ ]:
def batch_classify_steerings(analyzer, classifier, steerings_data: List[Dict], 
                           cache_file: str = "classification_cache.json",
                           batch_delay: float = 1.0) -> List[Dict]:
    """Batch process steerings with caching and rate limiting."""
    
    # Load existing cache
    cache_path = Path(cache_file)
    cache = {}
    if cache_path.exists():
        with open(cache_path, 'r') as f:
            cache = json.load(f)
        print(f"Loaded cache with {len(cache)} entries")
    
    results = []
    new_classifications = 0
    
    for i, steering in enumerate(tqdm(steerings_data, desc="Classifying steerings")):
        # Create cache key
        cache_key = f"{steering['model']}_{steering['dataset']}_{steering['experiment_hash']}_{steering['alpha']}_{hash(steering['steered_generation'])}"
        
        # Check cache first
        if cache_key in cache:
            classification = cache[cache_key]
        else:
            # Classify with O3
            classification = classifier.classify_reasoning(
                steering['question'],
                steering['steered_generation']
            )
            
            # Cache result
            cache[cache_key] = classification
            new_classifications += 1
            
            # Save cache every 10 new classifications
            if new_classifications % 10 == 0:
                with open(cache_path, 'w') as f:
                    json.dump(cache, f, indent=2)
            
            # Rate limiting
            time.sleep(batch_delay)
        
        # Combine steering data with classification
        result = {**steering, **classification}
        results.append(result)
    
    # Final cache save
    with open(cache_path, 'w') as f:
        json.dump(cache, f, indent=2)
    
    print(f"\nCompleted: {new_classifications} new classifications, {len(results)} total results")
    return results

print("✓ Batch processing function ready")

## Main Analysis: Load and Classify All Successful Steerings

In [ ]:
# Load all successful steerings (you can modify filters as needed)
all_steerings = analyzer.load_successful_steerings(
    # models=None,  # Include all models
    # datasets=None,  # Include all datasets  
    # alpha_range=None  # Include all alpha values
)

print(f"Total successful steerings to classify: {len(all_steerings)}")

# Show distribution
if all_steerings:
    df_overview = pd.DataFrame(all_steerings)
    
    print("\nDistribution by model:")
    print(df_overview['model'].value_counts())
    
    print("\nDistribution by dataset:")
    print(df_overview['dataset'].value_counts())
    
    print("\nDistribution by alpha value:")
    print(df_overview['alpha'].value_counts().sort_index())

In [ ]:
# Run batch classification (this may take a while depending on API limits)
# Start with a subset for testing
subset_size = min(50, len(all_steerings))  # Limit to 50 for initial testing
test_subset = all_steerings[:subset_size]

print(f"Starting classification of {len(test_subset)} samples...")

classified_results = batch_classify_steerings(
    analyzer, classifier, test_subset,
    cache_file="confabulation_classification_cache.json",
    batch_delay=2.0  # 2 second delay between API calls
)

## Analysis and Visualization

In [ ]:
# Convert results to DataFrame for analysis
if classified_results:
    df_results = pd.DataFrame(classified_results)
    
    print(f"Classification results: {len(df_results)} samples")
    
    # Classification distribution
    print("\nClassification distribution:")
    print(df_results['classification'].value_counts())
    
    # Confidence distribution
    print("\nConfidence distribution:")
    print(df_results['confidence'].value_counts())
    
    # Cross-tabulation: classification vs confidence
    print("\nClassification × Confidence:")
    print(pd.crosstab(df_results['classification'], df_results['confidence']))

In [ ]:
# Analysis by alpha value
if 'df_results' in locals() and len(df_results) > 0:
    # Filter out N/A classifications for main analysis
    df_clean = df_results[df_results['classification'] != 'N/A'].copy()
    
    if len(df_clean) > 0:
        # Classification rates by alpha value
        alpha_classification = pd.crosstab(df_clean['alpha'], df_clean['classification'], normalize='index') * 100
        
        print("\nClassification rates by alpha value (%)")
        print(alpha_classification.round(1))
        
        # Visualization
        fig, axes = plt.subplots(2, 2, figsize=(15, 10))
        fig.suptitle('Reasoning Classification Analysis', fontsize=16)
        
        # 1. Overall classification distribution
        df_clean['classification'].value_counts().plot(kind='bar', ax=axes[0,0], rot=45)
        axes[0,0].set_title('Classification Distribution')
        axes[0,0].set_ylabel('Count')
        
        # 2. Classification by alpha (stacked bar)
        alpha_classification.plot(kind='bar', stacked=True, ax=axes[0,1], rot=45)
        axes[0,1].set_title('Classification Rates by Alpha Value')
        axes[0,1].set_ylabel('Percentage')
        axes[0,1].legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        
        # 3. Confidence distribution
        df_clean['confidence'].value_counts().plot(kind='pie', ax=axes[1,0], autopct='%1.1f%%')
        axes[1,0].set_title('Confidence Distribution')
        
        # 4. Focus on non-entailment vs confabulation
        focus_classifications = df_clean[df_clean['classification'].isin(['non_entailment', 'confabulation'])]
        if len(focus_classifications) > 0:
            focus_alpha = pd.crosstab(focus_classifications['alpha'], focus_classifications['classification'], normalize='index') * 100
            focus_alpha.plot(kind='line', marker='o', ax=axes[1,1])
            axes[1,1].set_title('Non-entailment vs Confabulation by Alpha')
            axes[1,1].set_xlabel('Alpha Value')
            axes[1,1].set_ylabel('Percentage')
            axes[1,1].legend()
        else:
            axes[1,1].text(0.5, 0.5, 'No non-entailment or\nconfabulation cases found', 
                          ha='center', va='center', transform=axes[1,1].transAxes)
            axes[1,1].set_title('Non-entailment vs Confabulation')
        
        plt.tight_layout()
        plt.show()
    else:
        print("No valid classifications found for analysis")
else:
    print("No results available for analysis")

## Sample Classifications for Review

In [ ]:
# Show sample classifications for manual review
if 'df_results' in locals() and len(df_results) > 0:
    classifications_of_interest = ['non_entailment', 'confabulation']
    
    for classification in classifications_of_interest:
        samples = df_results[df_results['classification'] == classification]
        
        if len(samples) > 0:
            print(f"\n{'='*50}")
            print(f"SAMPLE {classification.upper()} EXAMPLES")
            print(f"{'='*50}")
            
            for i, (_, row) in enumerate(samples.head(3).iterrows()):
                print(f"\n--- Example {i+1} ---")
                print(f"Model: {row['model']}")
                print(f"Dataset: {row['dataset']}")
                print(f"Alpha: {row['alpha']}")
                print(f"Confidence: {row['confidence']}")
                print(f"\nQuestion: {row['question']}")
                print(f"\nSteered Generation: {row['steered_generation'][:400]}...")
                print(f"\nClassification Reasoning: {row['reasoning']}")
        else:
            print(f"\nNo examples found for {classification}")

## Export Results

In [ ]:
# Export results for further analysis
if 'df_results' in locals() and len(df_results) > 0:
    # Create output directory
    output_dir = Path("../results/confabulation_analysis")
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # Export full results
    df_results.to_csv(output_dir / "classified_steerings.csv", index=False)
    print(f"✓ Exported {len(df_results)} classified results to {output_dir / 'classified_steerings.csv'}")
    
    # Export summary statistics
    summary_stats = {
        'total_classifications': len(df_results),
        'classification_counts': df_results['classification'].value_counts().to_dict(),
        'confidence_counts': df_results['confidence'].value_counts().to_dict(),
        'na_rate': (df_results['classification'] == 'N/A').mean() * 100,
        'high_confidence_rate': (df_results['confidence'] == 'high').mean() * 100
    }
    
    with open(output_dir / "summary_stats.json", 'w') as f:
        json.dump(summary_stats, f, indent=2)
    
    print(f"✓ Exported summary statistics to {output_dir / 'summary_stats.json'}")
    print("\nSummary:")
    print(json.dumps(summary_stats, indent=2))
else:
    print("No results to export")

## Next Steps

This notebook provides a framework for analyzing confabulation and non-entailment in steered model outputs. To extend the analysis:

1. **Scale up**: Process larger datasets by adjusting the `subset_size` and managing API rate limits
2. **Cross-model comparison**: Include multiple models in the analysis
3. **Dataset-specific analysis**: Examine how different tasks affect confabulation rates
4. **Alpha strength analysis**: Study how stronger steering (higher |alpha|) affects reasoning patterns
5. **Validation**: Manual review of classifications to assess O3's accuracy
6. **Statistical testing**: Add significance tests for trends across alpha values

The caching system ensures that classifications are preserved between runs, making it efficient to iterate on the analysis.